# **Purpose of Notebook**
This notebook assesses the precision of *DetectOffice* function.

In [1]:
import sys, json, os, cv2
import pandas as pd

Year,Showa='1942','17'

origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)'
df = pd.read_csv(origin+'\Tokyo_Jobs\Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

In [2]:
from google.cloud import vision
from google.cloud.vision_v1 import types
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'\\Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'

In [3]:
origin+'\\Tokyo_Jobs\\Processed_Data\\Data_Collection\\Extract_Data'

'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Data_Collection\\Extract_Data'

# Experiment

In [9]:
StepAError,StepBError,StepCError,MainError=[],[],[],[]
Dict={}
for Page in range(10, 80, 5):    
    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/HQ/Page'+'{:03d}'.format(Page)+'.jpg'
    img=cv2.imread(os.path.join(origin,file_path))
    
    #Get Office Info: StepA
    try:        
        BookPage=2*Page-14
        Rows=df[(df['Page']==BookPage)]
        NxRows=df[(df['Page']==BookPage+1)]
        if len(Rows)==0:
            print('No Office at Right Side. Page'+str(BookPage))
        if len(NxRows)==0:
            print('No Office at Left Side. Page'+str(BookPage+1))
    except:
        StepAError.append(Page)
        continue
        
    #Apply OCR: StepB
    try:        
        sys.path.append(origin+'\\Tokyo_Jobs\\Data_Collection\\Extract_Data')
        from Read import Vision
        # Instantiates a client
        client = vision.ImageAnnotatorClient()
        texts=Vision(img,'zh',client)
    except:
        StepBError.append(Page)
        continue
        
    #Filter Data: StepC
    try:
        sys.path.append(origin+'\\Tokyo_Jobs\\Data_Collection\\Split_Page')
        from Split import VertPart
        sys.path.append(origin+'\\Tokyo_Jobs\\Data_Collection\\General')
        from PageCut import HoriPart
        from Organize import Filter, ConvertDict
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/HQ/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)

        HoriPoint=HoriPart(img,Page,texts)[0]

        try:
            VertPoint=VertPart(path)[1]
        except:
            print('Failed detecting Vertical Point')
            VertPoint=1300

        ##Right Page
        BoxR=Filter(BookPage,texts,HoriPoint)
        BoxR=ConvertDict(BoxR)

        ##Left Page
        BoxL=Filter(BookPage+1,texts,HoriPoint)
        BoxL=ConvertDict(BoxL)

    except:
        StepCError.append(Page)
        continue

        
    
    #Get Office Location: MainStep
    #from Detect import DetectOffice
    from Organize import FilterBox
    from Detect import DetectOffice
    LocList=[]
    try:        
        #RightPage
        OfficeList=Rows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxR, Office,VertPoint))
            BoxR=FilterBox(BoxR,LocList,VertPoint)

        #LeftPage
        OfficeList=NxRows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxL, Office,VertPoint))
            BoxL=FilterBox(BoxL,LocList,VertPoint)

        Dict[Page]=LocList
    except:
        MainError.append(Page)
        continue

No Office at Right Side. Page36
No Office at Right Side. Page46
No Office at Left Side. Page47
Horizontal Line was not automatically detected... Defining line arbitrariry...
Horizontal Line was not automatically detected... Defining line arbitrariry...
No Office at Left Side. Page87
No Office at Right Side. Page106
No Office at Left Side. Page107
No Office at Right Side. Page116
No Office at Left Side. Page117


# Summary of performance

**1. Non-Running Error**

In [10]:
from Show import Show
def ErrorRate(ErrorList,PageList):
    return(round(len(ErrorList)/len(range(10, 120, 5)),2))

In [11]:
ErrorRate(StepAError,range(10, 120, 5)),ErrorRate(StepBError,range(10, 120, 5)),ErrorRate(StepCError,range(10, 120, 5)),ErrorRate(MainError,range(10, 120, 5))

(0.0, 0.0, 0.05, 0.05)

In [12]:
MainError

[40]

**Results** 

Causes of Main Error
- Cannot find words (Old japanese letter) (0).
- Word not found (1).

**Results**: 5% of pages did not go through

**2.Miss Assignment Error**

In [13]:
for Page in list(Dict.keys()):
    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/HQ/Page'+'{:03d}'.format(Page)+'.jpg'
    img=cv2.imread(os.path.join(origin,file_path))
    LocList=Dict[Page]
    if len(LocList)!=0:
        for n in range(len(LocList)):
            print(Page,LocList[n]['OfficeName'],LocList[n]['Words'])
        Show(img,LocList)

10 議案課 議案課福利
10 区政課 政信鈴木美子
10 報道課 報道課
10 統計課 統計九
15 用品課 用品課主事
15 購買課 購買課
20 動員課 動員課
20 貯蓄奨励課 貯蓄獎勵課隆
20 商工課 商工課
25 體力課 體力課
25 衛生課 衛生課技師
35 大久保病院 大久保病院淀橋區東大久保
35 大塚病院 大塚病院小石川
45 庶務課 庶務課
45 計書課 計畫課
50 治水課 治水課
55 葛飾区出張所 葛飾區出張所(本田大
55 江戸川区出張所 江戸川區出張所(城31四
55 庶務課 庶務課
55 築港課 築港課
70 調査課 調查課
70 教習所 教習所(
75 車輛工場 車輛工場(三
75 三田電車営業所 三田電車營業所(三


- Cause of Errors
    1. Reading Names of office at right/left top corners: (0)

    2. Confused with names: (0)
    
        <= Precision should increase by better line detection.
        
    3. Searching in areas of previous office: (0)
        
    4. Wrong Office Name: (0)
        
    5. Wrong Office Page: (0)
    
    6. Word not found in page: (1)